# 01 · Iceberg data checks and initial EDA

第3–4步：检查真实数据并解释基础分布。本Notebook不训练模型、不填补角度、不生成正式fold。

先在仓库根目录运行 `python -m pip install -r requirements-eda.txt`，选择同一个Python解释器，将三个**解压后的实际文件**放在 `data/raw/`。

分析实现位于 `src/eda.py`。阅读代码并复运行后，再把你自己的理解写入报告；AI协助情况见 `docs/AI_USAGE.md`。

In [ ]:
from pathlib import Path
import sys
from IPython.display import Image, display

# Works when launched from either the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "eda.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cs4486-iceberg repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "raw"  # Edit this path if your local data live elsewhere.
RESULTS_DIR = ROOT / "results" / "eda"
METADATA_DIR = ROOT / "outputs" / "eda"
print("Repository:", ROOT.name)
print("Data directory:", DATA_DIR.relative_to(ROOT) if DATA_DIR.is_relative_to(ROOT) else DATA_DIR)

## 1. Read and validate the data

`analyze` streams JSON records, validates both radar bands, and calculates exact image fingerprints. Only metadata and a few sample images stay in memory. The larger test file may take some time to scan.

No raw pixel values are divided by 255. Missing angles remain missing at this stage.

In [ ]:
from src.eda import analyze, save_results
import pandas as pd

bundle = analyze(DATA_DIR)
summary = bundle["summary"]
train = bundle["train"]["metadata"]
test = bundle["test"]["metadata"]
pd.DataFrame({split: summary[split] for split in ("train", "test")}).T

## 2. Labels and missing angles

0 means ship; 1 means iceberg. Examine the class counts before deciding whether imbalance handling is needed. Missingness can be associated with labels without being a reliable property of future data.

In [ ]:
labels = train["is_iceberg"].value_counts().sort_index()
display(pd.DataFrame({"count": labels, "share": labels / len(train)}))
display(pd.crosstab(train["inc_angle"].isna().rename("angle_missing"),
                    train["is_iceberg"], dropna=False))
print("Test missing angles:", test["inc_angle"].isna().sum())

## 3. Repeated angles and exact image duplicates

Repeated angle values are a possible source-association signal, not verified scene IDs. Mixed-label angle groups must not be split by label to improve a score. Exact image fingerprints do not detect near duplicates.

In [ ]:
groups = (train.dropna(subset=["inc_angle"]).groupby("inc_angle")
          .agg(samples=("id", "size"), classes=("is_iceberg", "nunique")))
repeated = groups[groups["samples"] > 1]
print("Repeated angle groups:", len(repeated))
print("Samples in those groups:", repeated["samples"].sum())
print("Single-class groups:", (repeated["classes"] == 1).sum())
print("Mixed-class groups:", (repeated["classes"] > 1).sum())
print("Exact image duplicate excess rows:", summary["train"]["exact_duplicate_image_excess_rows"])
print("Train/test overlap:", summary["overlap"])

## 4. Submission format

The example CSV is a template. Its 0.5 values are placeholders, not model predictions. A changed row order is reported separately from missing or mismatched IDs.

In [ ]:
summary["submission"]

## 5. Save and inspect the EDA figure

The plot uses the first available ship and iceberg as illustrative examples. The shared percentile display scale affects visualization only. Train and test angle histograms use density so their different dataset sizes do not dominate the comparison.

Small aggregate results and the figure go to `results/eda/`. Row-level metadata go to ignored `outputs/eda/`.

In [ ]:
save_results(bundle, RESULTS_DIR, METADATA_DIR)
from IPython.display import Image, display
display(Image(filename=str(RESULTS_DIR / "overview.png"), width=1000))

## 6. Team interpretation and next step

- Does the label distribution justify weighting the loss? Explain your choice.
- All missing-angle training records are ships, but test angles are present. How could a missingness indicator distort a validation comparison?
- Why are exact angle groups useful to investigate, yet insufficient to prove scene independence?
- Near-duplicate inspection remains unfinished. Decide how to review candidate matches before freezing folds.

Read `docs/VALIDATION_PLAN.md` for the proposed fifth step. The teacher's unified split takes priority if provided. No folds or model scores are produced here.